In [2]:
data_root_path = "/media/data/minhht/moe_plantdeasse/data/tomato_only"

In [3]:
import os

print(os.listdir(data_root_path))

['Tomato mold leaf', 'Tomato leaf yellow virus', 'Tomato leaf mosaic virus', 'Tomato leaf bacterial spot', 'Tomato Early blight leaf', 'Tomato leaf', 'Tomato Septoria leaf spot', 'Tomato leaf late blight']


In [4]:
"""
Utility Functions and Dataset Loading Classes.

This module provides helper functions and dataset classes for loading and processing
plant disease images. It includes the LoadDataset class for structured data loading
with support for train/validation/test splits.

Key Features:
    - Automatic dataset splitting with stratification
    - Class to index mapping for label encoding
    - Image loading with PIL
    - Transform pipeline support via Albumentations
"""

import os
from typing import List, Tuple, Dict, Literal, Optional
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset
from torchvision.transforms import transforms

from PIL import Image
from sklearn.model_selection import train_test_split


class LoadDataset(Dataset):
    """
    Dataset loader for plant disease classification.
    
    Loads images from a structured directory of disease classes and provides
    automatic train/validation/test splitting with stratification to ensure
    balanced class distribution across splits.
    
    The dataset expects a directory structure like:
        root_dir/
        ├── Tomato_Bacterial_spot/
        ├── Tomato_Early_blight/
        ├── Tomato_healthy/
        └── ... (other disease classes)
    
    Attributes:
        root_dir (Path): Root directory containing disease class folders
        split (str): Dataset split ('train', 'val', or 'test')
        train_ratio (float): Proportion of data for training
        image_paths (List[str]): List of image file paths for the selected split
        labels (List[int]): Class labels corresponding to image_paths
        class_to_idx (Dict[str, int]): Mapping from class name to class index
        idx_to_class (Dict[int, str]): Mapping from class index to class name
    """

    def __init__(
        self,
        root_dir: Path,
        split: Literal['train', 'validation', 'test'],
        train_ratio: float = 0.8,
        transform: transforms.Compose = None
    ) -> None:
        """
        Initialize the dataset loader.
        
        Args:
            root_dir (Path): Root directory containing class subdirectories
            split (str, optional): Dataset split - 'train', 'val', or 'test'. Defaults to 'train'.
            train_ratio (float, optional): Proportion of data for training (0 to 1). Defaults to 0.8.
            transform (transforms.Compose, optional): Image transformation pipeline. Defaults to None.
        """
        self.root_dir = root_dir
        self.transform = transform
        self.split = split
        self.train_ratio = train_ratio
        self.image_paths, self.labels, self.class_to_idx, self.idx_to_class = self._split_dataset()

    def _load_image(self, root_dir: Path) -> Tuple[List[str], List[int], Dict[str, int], Dict[int, str]]:
        """
        Load all images and labels from the root directory.
        
        Scans the root directory for subdirectories starting with "Tomato" (disease classes),
        collects all image files from each class, and creates class-to-index mappings.
        
        Args:
            root_dir (Path): Root directory containing class subdirectories
            
        Returns:
            Tuple containing:
                - image_paths (List[str]): Absolute paths to all image files
                - labels (List[int]): Class labels (0-indexed) corresponding to each image
                - class_to_idx (Dict[str, int]): Mapping from class name to class index
                - idx_to_class (Dict[int, str]): Mapping from class index to class name
        """
        class_names = sorted(
            [d for d in os.listdir(root_dir)
             if os.path.isdir(os.path.join(root_dir, d))
             and d.startswith("Tomato")]
        )
        class_to_idx = {class_name: idx for idx, class_name in enumerate(class_names)}
        idx_to_class = {idx: class_name for class_name, idx in class_to_idx.items()}
        image_paths = []
        labels = []
        for class_name in class_names:
            dir = os.path.join(root_dir, class_name)
            for fname in os.listdir(dir):
                if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
                    image_paths.append(os.path.join(dir, fname))
                    labels.append(class_to_idx[class_name])
        return image_paths, labels, class_to_idx, idx_to_class

    def _split_dataset(self) -> Tuple[List[str], List[int], Dict[str, int], Dict[int, str]]:
        """
        Split dataset into train, validation, and test sets.
        
        Uses stratified sampling to ensure balanced class distribution across all splits:
        - 80% training, 20% temporary (from train_ratio)
        - Temporary split: 50% validation, 50% test
        
        Returns:
            Tuple containing:
                - image_paths (List[str]): Image paths for the selected split
                - labels (List[int]): Labels for the selected split
                - class_to_idx (Dict[str, int]): Class name to index mapping
                - idx_to_class (Dict[int, str]): Index to class name mapping
                
        Raises:
            ValueError: If split is not 'train', 'val', or 'test'
        """
        image_paths, labels, class_to_idx, idx_to_class = self._load_image(self.root_dir)

        train_paths, temp_paths, train_labels, temp_labels = train_test_split(
            image_paths, labels, test_size= 1-self.train_ratio, stratify=labels, random_state=42, shuffle=True
        )

        val_paths, test_paths, val_labels, test_labels = train_test_split(
            temp_paths, temp_labels, test_size=0.5, stratify=temp_labels, random_state=42, shuffle=True
        )

        if self.split == 'train':
            return train_paths, train_labels, class_to_idx, idx_to_class
        elif self.split == 'validation':
            return val_paths, val_labels, class_to_idx, idx_to_class
        elif self.split == 'test':
            return test_paths, test_labels, class_to_idx, idx_to_class
        else:
            raise ValueError("split must be 'train', 'validation', or 'test'")

    def __len__(self) -> int:
        """
        Return the total number of samples in this dataset split.
        
        Returns:
            int: Number of images in the current split
        """
        return len(self.image_paths)
    
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        """
        Get a single sample from the dataset by index.
        
        Loads an image from disk, applies transformations if specified, and returns
        the transformed image tensor and its corresponding class label.
        
        Args:
            idx (int): Index of the sample to retrieve
            
        Returns:
            Tuple containing:
                - image (torch.Tensor): Transformed image tensor
                - label (int): Class label (0-indexed)
        """
        image_path = self.image_paths[idx]
        label = self.labels[idx]
        image = Image.open(image_path).convert('RGB')
        image = np.array(image)
        if self.transform:
            augumented = self.transform(image=image)
            image = augumented["image"]
        return image, label


In [5]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

transform_pipeline = A.Compose([
    A.Resize(224, 224),
    A.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
    ToTensorV2()
])

train_dataset = LoadDataset(data_root_path, split='train', train_ratio=0.8, transform=transform_pipeline)
val_dataset = LoadDataset(data_root_path, split='validation', train_ratio=0.8, transform=transform_pipeline)
test_dataset = LoadDataset(data_root_path, split='test', train_ratio=0.8, transform=transform_pipeline)

print("Number of training samples:", len(train_dataset))
print("Number of validation samples:", len(val_dataset))
print("Number of test samples:", len(test_dataset))
print("Class to index mapping:", train_dataset.class_to_idx)

/media/data/minhht/moe_plantdeasse/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of training samples: 2274
Number of validation samples: 284
Number of test samples: 285
Class to index mapping: {'Tomato Early blight leaf': 0, 'Tomato Septoria leaf spot': 1, 'Tomato leaf': 2, 'Tomato leaf bacterial spot': 3, 'Tomato leaf late blight': 4, 'Tomato leaf mosaic virus': 5, 'Tomato leaf yellow virus': 6, 'Tomato mold leaf': 7}


In [6]:
from torch.utils.data import DataLoader
from torchvision import transforms


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [7]:
import torch
import torch.nn as nn
from torchvision import models

model = models.mobilenet_v3_large(pretrained=True)

# thay classifier
in_features = model.classifier[3].in_features
model.classifier[3] = nn.Linear(in_features, 8)

/media/data/minhht/moe_plantdeasse/venv/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/media/data/minhht/moe_plantdeasse/venv/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Large_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [8]:
# criterion = nn.CrossEntropyLoss()

# optimizer = torch.optim.AdamW(
#     model.parameters(),
#     lr=0.001,
# )

# num_epochs = 50
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)

# for epoch in range(num_epochs):
#     model.train()
#     total_loss = 0
#     for images, labels in train_loader:
#         images, labels = images.to(device), labels.to(device)
#         optimizer.zero_grad()
#         outputs = model(images)
#         loss = criterion(outputs, labels)
#         loss.backward()
#         optimizer.step()
#         total_loss += loss.item()
#     avg_loss = total_loss / len(train_loader)
#     print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

In [9]:
torch.save(model.state_dict(), "mobilenetv3_large.pth")

In [10]:
# load checkpoint
checkpoint_path = "/media/data/minhht/moe_plantdeasse/notebook/mobilenetv3_large.pth"
model.load_state_dict(torch.load(checkpoint_path, map_location="cuda"))

# bỏ classifier
model.classifier = nn.Identity()

model.eval()

/media/data/minhht/moe_plantdeasse/venv/lib/python3.11/site-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  warnings.warn(


MobileNetV3(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        )
      )
    )
    (2): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 64, kernel_size=(1, 1), stride=(1, 1), bi

In [11]:
input_dummy = torch.randn(10, 3, 224, 224)
output_dummy = model(input_dummy)
print("Output shape:", output_dummy.shape)

Output shape: torch.Size([10, 960])


In [12]:
all_features = []
all_labels = []

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
with torch.no_grad():
    for images, labels in train_loader:
        images = images.to(device)
        features = model(images)
        all_features.append(features.cpu())
        all_labels.append(labels.cpu())

In [13]:
features = torch.cat(all_features)
labels = torch.cat(all_labels)

In [14]:
features.shape, labels.shape

(torch.Size([2274, 960]), torch.Size([2274]))

In [15]:
from sklearn.preprocessing import StandardScaler

sc = StandardScaler()
feature_normalized = sc.fit_transform(features)

In [16]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
features_pca = pca.fit_transform(feature_normalized)
features_pca.shape

(2274, 2)

In [17]:
import skfuzzy as fuzz


cntr, u, u0, d, jm, p, fpc = fuzz.cluster.cmeans(
    features_pca.T,
    c = 3,
    m = 2.0,
    error = 1e-9,
    maxiter = 3000,
    init = None
)

In [18]:
from sklearn.metrics import silhouette_score
import numpy as np

label_fcm = np.argmax(u, axis=0)

score = silhouette_score(features_pca, label_fcm)
print(score)

0.36417542174392553


In [19]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42)
clusters_labels_kmeans = kmeans.fit_predict(features_pca)

In [20]:
from sklearn.metrics import silhouette_score

score = silhouette_score(features_pca, clusters_labels_kmeans)
print("Silhouette:", score)

Silhouette: 0.36795117415910916


In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=42)
features_tsne = tsne.fit_transform(feature_normalized)


### fcm_clustering
cntr, u, u0, d, jm, p, fpc = fuzz.cluster.cmeans(
    features_tsne.T,
    c = 3,
    m = 2.0,
    error = 1e-9,
    maxiter = 3000,
    init = None
)
label_fcm = np.argmax(u, axis=0)

score = silhouette_score(features_tsne, label_fcm)
print("Silhouette_fcm:", score)

### kmeans_clustering
kmeans = KMeans(n_clusters=3, random_state=42)
clusters_labels_kmeans = kmeans.fit_predict(features_tsne)
score = silhouette_score(features_tsne, clusters_labels_kmeans)
print("Silhouette_kmeans:", score)

Silhouette_fcm: 0.4097541272640228
Silhouette_kmeans: 0.41424843668937683


: 

In [23]:
# cluster_labels = np.argmax(u, axis=0)

In [24]:
# clusters = {}

# for i in range(3):
#     idx = np.where(cluster_labels == i)[0]

#     X_cluster = features[idx]
#     y_cluster = labels[idx]   # nhãn sâu bệnh

#     clusters[i] = (X_cluster, y_cluster)

In [25]:
# from collections import Counter

# for i in range(3):
#     idx = np.where(cluster_labels == i)[0]

#     cluster_labels_true = labels[idx]
#     cluster_labels_true = cluster_labels_true.numpy()

#     print(f"Cluster {i}:")
#     print(Counter(cluster_labels_true))